# Workshop 2 - Exploring the Silver layer

This notebook reads the Parquet files that the `silver_processing` DAG wrote to `datalake_silver/` and checks what the preprocessing produced: schema and types, null rates, outlier flags, lineage back to the Bronze JSON files, the quarantine, and the tokens extracted from the scraped USGS notices.

Run it from the repository root (or from `notebooks/`) after the DAGs have run at least once.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "datalake_silver").exists() else Path.cwd().parent
SILVER, BRONZE = ROOT / "datalake_silver", ROOT / "datalake_bronze"
pd.set_option("display.max_columns", 30, "display.width", 160)

files = sorted(SILVER.glob("*.parquet"))
inventory = pd.DataFrame({
    "file": [f.name for f in files],
    "topic": [f.name.rsplit("_", 2)[0] for f in files],
    "rows": [len(pd.read_parquet(f)) for f in files],
    "kb": [round(f.stat().st_size / 1024, 1) for f in files],
})
inventory

,file,topic,rows,kb
0,alert_notices_20261116_224153.parquet,alert_notices,54,20.3
1,earthquakes_20261116_224153.parquet,earthquakes,229,47.0
2,eruptions_20261116_224153.parquet,eruptions,1281,127.3
3,volcano_updates_20261116_224153.parquet,volcano_updates,9,37.4
4,volcanoes_20261116_224153.parquet,volcanoes,165,29.2


## 1. Read one topic: all files of a topic as one DataFrame

Every file of a topic has the same schema, so they concatenate cleanly (the same property lets PySpark read the folder in Workshop 3).

In [2]:
def read_topic(topic: str) -> pd.DataFrame:
    parts = [pd.read_parquet(f) for f in sorted(SILVER.glob(f"{topic}_*.parquet"))]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

topics = ["earthquakes", "alert_notices", "volcanoes", "eruptions", "volcano_updates"]
data = {t: read_topic(t) for t in topics}
{t: df.shape for t, df in data.items()}

{'earthquakes': (229, 22),
 'alert_notices': (54, 16),
 'volcanoes': (165, 13),
 'eruptions': (1281, 15),
 'volcano_updates': (9, 25)}

In [3]:
eq = data["earthquakes"]
eq.dtypes

record_id                               str
content_hash                            str
event_time              datetime64[ns, UTC]
updated_time            datetime64[ns, UTC]
latitude                            float64
longitude                           float64
depth_km                            float64
mag                                 float64
mag_type                           category
network                            category
review_status                      category
place                                   str
nearest_vnum                          Int64
distance_km                         float64
mag_missing                            bool
is_outlier_mag_iqr                     bool
is_outlier_mag_z                       bool
is_outlier_depth_iqr                   bool
is_outlier_depth_z                     bool
bronze_file                             str
bronze_sha256                           str
processed_utc           datetime64[ns, UTC]
dtype: object

In [4]:
eq.head()

,record_id,content_hash,event_time,updated_time,latitude,longitude,depth_km,mag,mag_type,network,review_status,place,nearest_vnum,distance_km,mag_missing,is_outlier_mag_iqr,is_outlier_mag_z,is_outlier_depth_iqr,is_outlier_depth_z,bronze_file,bronze_sha256,processed_utc
0,usgs:nc75464572,456e0833580340ebf5fb48fd428fcc5fbca4b17f74b008...,2026-11-09 22:45:37.890+00:00,2026-11-09 22:47:12.406+00:00,38.799000,-122.797997,0.160000,1.05,md,nc,automatic,"4 km NW of The Geysers, CA",323100,19.17,False,False,False,False,False,usgs_feed_20261116_223954.json,3651e8e55a5fed0860074b91c74f12a2791fc69ea1711e...,2026-11-16 22:41:53.156212+00:00
1,usgs:nc75464587,f25f5f8bdd57c15d5fcbe82c4aa686cc3c281fe1e0ebaa...,2026-11-09 23:55:18.600+00:00,2026-11-09 23:56:57.049+00:00,38.768166,-122.940834,4.010000,1.02,md,nc,automatic,"8 km ESE of Cloverdale, CA",323100,26.88,False,False,False,False,False,usgs_feed_20261116_223954.json,3651e8e55a5fed0860074b91c74f12a2791fc69ea1711e...,2026-11-16 22:41:53.156212+00:00
2,usgs:hv75068917,c480fcfbc72f96d12804ecde7a78e10c079aa5207bfaa3...,2026-11-10 01:49:48.400+00:00,2026-11-10 02:55:57.040+00:00,19.191667,-155.370333,28.730000,2.58,ml,hv,reviewed,"11 km E of Pāhala, Hawaii",332010,26.96,False,False,False,False,True,usgs_feed_20261116_223954.json,3651e8e55a5fed0860074b91c74f12a2791fc69ea1711e...,2026-11-16 22:41:53.156212+00:00
3,usgs:hv75068947,9eab9bf2d4aeb76909e1f03904a31e477c773fe56077c7...,2026-11-10 05:01:51.090+00:00,2026-11-10 05:05:16.840+00:00,19.335501,-155.321671,31.709999,1.70,md,hv,automatic,"15 km SW of Volcano, Hawaii",332010,10.18,False,False,False,False,True,usgs_feed_20261116_223954.json,3651e8e55a5fed0860074b91c74f12a2791fc69ea1711e...,2026-11-16 22:41:53.156212+00:00
4,usgs:nc75464677,0666f883b82cf2acd931e4d6c706d66ca8ac1b38fd60d0...,2026-11-10 05:45:36.030+00:00,2026-11-10 05:47:12.552+00:00,38.834332,-122.816666,2.030000,1.08,md,nc,automatic,"8 km NW of The Geysers, CA",323100,15.62,False,False,False,False,False,usgs_feed_20261116_223954.json,3651e8e55a5fed0860074b91c74f12a2791fc69ea1711e...,2026-11-16 22:41:53.156212+00:00


## 2. Null rates per field

Nulls that remain after Silver are expected and documented: missing VEI, unknown last eruption year, HANS volcanoes outside the GVP Holocene list (no coordinates), and optional notice fields.

In [5]:
null_rates = pd.concat({
    t: (df.drop(columns=["tokens"], errors="ignore").isna().mean() * 100).round(1)
    for t, df in data.items() if not df.empty
}).rename("null_%")
null_rates[null_rates > 0].to_frame()

null_%
alert_notices   latitude               7.4
                longitude              7.4
volcanoes       last_eruption_year    26.7
eruptions       start_time            30.2
                vei                   37.5
volcano_updates volcano_name          11.1
                vnum                  55.6
                issued_raw            33.3
                notice_number         88.9
                notice_url            66.7

## 3. Earthquakes: ranges and outlier flags

Outliers are flagged, not removed: at volcanoes, swarms and very shallow or above-sea-level hypocentres are the signal.

In [6]:
eq[["mag", "depth_km", "distance_km"]].describe().round(2)

,mag,depth_km,distance_km
count,229.00,229.00,229.00
mean,1.60,22.55,17.93
std,0.51,37.63,8.29
min,1.00,-2.74,0.18
25%,1.19,2.21,14.65
50%,1.56,5.00,18.21
75%,1.89,26.87,25.40
max,4.90,219.57,30.00


In [7]:
flags = [c for c in eq.columns if c.startswith("is_outlier_")] + ["mag_missing"]
eq[flags].sum().rename("flagged rows").to_frame()

,flagged rows
is_outlier_mag_iqr,3
is_outlier_mag_z,2
is_outlier_depth_iqr,31
is_outlier_depth_z,62
mag_missing,0


In [8]:
eq.loc[eq["is_outlier_mag_iqr"] | eq["is_outlier_depth_z"],
       ["record_id", "event_time", "mag", "mag_type", "depth_km", "nearest_vnum", "distance_km", "place"]].head(10)

,record_id,event_time,mag,mag_type,depth_km,nearest_vnum,distance_km,place
2,usgs:hv75068917,2026-11-10 01:49:48.400+00:00,2.58,ml,28.730000,332010,26.96,"11 km E of Pāhala, Hawaii"
3,usgs:hv75068947,2026-11-10 05:01:51.090+00:00,1.70,md,31.709999,332010,10.18,"15 km SW of Volcano, Hawaii"
6,usgs:aka2026whamdt,2026-11-10 11:49:22.205+00:00,2.20,ml,134.900000,313020,13.64,"68 km WNW of Happy Valley, Alaska"
14,usgs:hv75069152,2026-11-10 17:08:20.500+00:00,1.72,md,31.549999,332010,19.57,"12 km NE of Pāhala, Hawaii"
16,usgs:hv75069057,2026-11-10 10:55:13.690+00:00,2.11,ml,30.810000,332000,18.21,"20 km SSE of Pāhala, Hawaii"
17,usgs:aka2026whtptc,2026-11-10 21:23:46.842+00:00,2.30,ml,143.800000,313020,15.20,"49 km ENE of Pedro Bay, Alaska"
19,usgs:aka2026whuxqh,2026-11-10 22:01:26.383+00:00,2.30,ml,84.500000,313030,28.32,"51 km WSW of Salamatof, Alaska"
23,usgs:aka2026wiftvu,2026-11-11 03:26:54.195+00:00,2.30,ml,91.100000,313030,20.65,"59 km W of Cohoe, Alaska"
25,usgs:hv75069327,2026-11-11 09:51:01.510+00:00,1.70,md,26.530001,332010,28.54,"13 km ESE of Pāhala, Hawaii"
30,usgs:aka2026wjbzzm,2026-11-11 14:33:31.803+00:00,2.20,ml,113.000000,313020,17.23,"59 km W of Happy Valley, Alaska"


In [9]:
vol = data["volcanoes"][["vnum", "name"]]
(eq.merge(vol, left_on="nearest_vnum", right_on="vnum", how="left")
   .groupby("name").size().sort_values(ascending=False).head(10).rename("earthquakes (7 days, <= 30 km)").to_frame())

,"earthquakes (7 days, <= 30 km)"
name,
Clear Lake Volcanic Field,74
Kilauea,23
Iliamna,14
Soda Lakes,14
Redoubt,13
Kamaehuakanaloa,12
Spurr,7
Mammoth Mountain,6
Katmai,6


## 4. Alerts, eruptions and volcanoes

In [10]:
al = data["alert_notices"]
pd.crosstab(al["alert_level"], al["color_code"])

color_code,GREEN,ORANGE,YELLOW
alert_level,,,
ADVISORY,0,0,2
NORMAL,51,0,0
WATCH,0,1,0


In [11]:
er = data["eruptions"]
summary = {
    "eruptions (US)": len(er),
    "VEI missing": int(er["vei_missing"].sum()),
    "BCE (no timestamp)": int((er["start_year"] < 1).sum()),
}
summary | er["time_precision"].value_counts().to_dict()

{'eruptions (US)': 1281,
 'VEI missing': 480,
 'BCE (no timestamp)': 387,
 'day': 657,
 'bce': 387,
 'year': 232,
 'month': 5}

## 5. Scraped USGS notices: text preprocessing

In [12]:
vu = data["volcano_updates"]
vu[["record_type", "observatory", "volcano_name", "alert_level", "as_of_time", "token_count"]]

,record_type,observatory,volcano_name,alert_level,as_of_time,token_count
0,elevated,hvo,Kilauea,ADVISORY,2026-11-16 17:26:55+00:00,20
1,elevated,avo,Great Sitkin,WATCH,2026-11-13 21:04:48+00:00,15
2,elevated,nmi,Ahyi Seamount,ADVISORY,2026-11-13 01:04:55+00:00,13
3,observatory_notice,avo,Katmai,NORMAL,2026-11-16 19:48:00+00:00,132
4,observatory_notice,calvo,NaN,NORMAL,2026-11-10 17:57:00+00:00,136
5,observatory_notice,cvo,CASCADE RANGE,NORMAL,2026-11-13 22:53:00+00:00,59
6,observatory_notice,hvo,KILAUEA,ADVISORY,2026-11-16 17:26:00+00:00,582
7,observatory_notice,nmi,AHYI SEAMOUNT,ADVISORY,2026-11-13 01:04:00+00:00,67
8,observatory_notice,yvo,YELLOWSTONE,NORMAL,2026-11-02 16:09:00+00:00,176


In [13]:
from collections import Counter
Counter(t for toks in vu["tokens"] for t in toks).most_common(20)

[('volcanic', 20),
 ('activity', 16),
 ('episode', 15),
 ('eruption', 14),
 ('fountaining', 13),
 ('november', 13),
 ('summit', 13),
 ('lava', 11),
 ('earthquakes', 11),
 ('recorded', 11),
 ('volcanoes', 10),
 ('data', 9),
 ('mount', 9),
 ('vents', 9),
 ('ground', 9),
 ('eruptive', 9),
 ('geyser', 9),
 ('kīlauea', 8),
 ('level', 8),
 ('ash', 8)]

In [14]:
vu[[c for c in vu.columns if c.startswith("kw_")]].sum().rename("notices mentioning").to_frame()

,notices mentioning
kw_eruption,6
kw_ash,2
kw_lava,2
kw_seismicity,4
kw_deformation,3
kw_gas,1
kw_plume,3


## 6. Lineage: every Silver row points to its Bronze file

In [15]:
lineage = pd.concat([df[["bronze_file", "bronze_sha256", "processed_utc"]].assign(topic=t)
                     for t, df in data.items() if not df.empty])
lineage.groupby(["topic", "bronze_file"]).size().rename("rows").to_frame()

,,rows
topic,bronze_file,
alert_notices,usgs_hans_20261116_223832.json,54
earthquakes,usgs_feed_20261116_223954.json,229
eruptions,gvp_eruptions_20261116_223856.json,1281
volcano_updates,usgs_volcano_updates_20261116_223834.json,9
volcanoes,gvp_volcanoes_20261116_223828.json,165


In [16]:
# open the Bronze file behind the first earthquake row
first = eq.iloc[0]
env = json.loads((BRONZE / first["bronze_file"]).read_text(encoding="utf-8"))
{k: env[k] for k in ("source_id", "retrieved_utc", "url", "record_count", "content_sha256")}

{'source_id': 'usgs_feed',
 'retrieved_utc': '2026-11-16T22:39:54.684152+00:00',
 'url': 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_week.geojson',
 'record_count': 1814,
 'content_sha256': 'fc07dc61a9a8d4ac18ebcc3d817600d034ed50aaa1625a127f0660008418ee8f'}

## 7. Quarantine

In [17]:
q_files = sorted((SILVER / "quarantine").glob("rejected_*.parquet"))
q = pd.concat([pd.read_parquet(f) for f in q_files], ignore_index=True) if q_files else pd.DataFrame()
q.groupby(["source_id", "reason"]).size().rename("records").to_frame() if not q.empty else "no rejected records" 

records
source_id reason                      
usgs_feed non_tectonic_type        131
usgs_hans no_notice_issued          13
          region_level_notice        2

In [18]:
q.head(3) if not q.empty else None

,source_id,reason,payload,bronze_file,processed_utc
0,usgs_feed,non_tectonic_type,"{""type"": ""Feature"", ""properties"": {""mag"": 1.19...",usgs_feed_20261116_223830.json,2026-11-16 22:41:53.156212+00:00
1,usgs_hans,region_level_notice,"{""volcano_name"": ""Alaskan Volcanoes"", ""vnum"": ...",usgs_hans_20261116_223832.json,2026-11-16 22:41:53.156212+00:00
2,usgs_hans,no_notice_issued,"{""volcano_name"": ""Isanotski"", ""vnum"": ""311370""...",usgs_hans_20261116_223832.json,2026-11-16 22:41:53.156212+00:00
